# UsedCarAI — 05 Model Comparison



In [ ]:
from pathlib import Path
import os
import sys

# Discover CPE-306 when opened from the project or UsedcarAi directory.
# Set USEDCAR_PROJECT_DIR to use a different checkout location.
_candidates = (Path.cwd(), *Path.cwd().parents)
_default_project = next(
    (p for p in _candidates if (p / "dataset").is_dir() and (p / "app").is_dir()),
    Path("/home/regen/Works/CPE-306"),
)
PROJECT_DIR = Path(os.environ.get("USEDCAR_PROJECT_DIR", _default_project)).expanduser().resolve()
DATASET_DIR = Path(os.environ.get("DATASET_DIR", PROJECT_DIR / "dataset")).expanduser().resolve()
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")
print("Python:", sys.executable)
print("Dataset:", DATASET_DIR)
print("Use the Python kernel from the project's ML environment.")


In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

warnings.filterwarnings("ignore")

ROOT = DATASET_DIR
TEST_PATH = ROOT / "data/processed/test.csv"
PRED_DIR = ROOT / "results/predictions"
METRICS_DIR = ROOT / "results/metrics"
FIGURES_DIR = ROOT / "results/figures"

for d in [METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

files = {
    "test": TEST_PATH,
    "rf": PRED_DIR / "random_forest_test_predictions.csv",
    "lr": PRED_DIR / "linear_regression_test_predictions.csv",
    "hy": PRED_DIR / "hybrid_test_predictions.csv",
}
for name, p in files.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing {name}: {p}")

print("All required files found.")

## Step 1 — Load Files

In [ ]:
test_df = pd.read_csv(files["test"])
rf_df = pd.read_csv(files["rf"])
lr_df = pd.read_csv(files["lr"])
hy_df = pd.read_csv(files["hy"])

print("Test:", test_df.shape)
print("RF:", rf_df.shape)
print("LR:", lr_df.shape)
print("Hybrid:", hy_df.shape)

## Step 2 — Verify Same Test Set

In [ ]:
test_ids = set(test_df["listing_id"].astype(str))
checks = {
    "RF": set(rf_df["listing_id"].astype(str)) == test_ids and len(rf_df) == len(test_df),
    "LR": set(lr_df["listing_id"].astype(str)) == test_ids and len(lr_df) == len(test_df),
    "Hybrid": set(hy_df["listing_id"].astype(str)) == test_ids and len(hy_df) == len(test_df),
}

for k, v in checks.items():
    print(k, "matches test:", v)

if not all(checks.values()):
    raise ValueError("Test set mismatch detected")

print("PASS: ทั้ง 3 วิธีใช้ Test set ชุดเดียวกัน")

## Step 3 — Recompute Metrics

In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return float(100*np.mean(2*np.abs(y_pred[mask]-y_true[mask])/denom[mask]))

def metrics(y_true, y_pred):
    return {
        "MAE_THB": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_THB": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "sMAPE_percent": smape(y_true, y_pred),
        "MedianAE_THB": float(median_absolute_error(y_true, y_pred)),
    }

base = test_df[["listing_id", "price"]].copy()

all_pred = (
    base
    .merge(rf_df[["listing_id", "predicted_price"]].rename(columns={"predicted_price":"rf_pred"}), on="listing_id")
    .merge(lr_df[["listing_id", "predicted_price"]].rename(columns={"predicted_price":"lr_pred"}), on="listing_id")
    .merge(hy_df[["listing_id", "hybrid_predicted_price"]].rename(columns={"hybrid_predicted_price":"hy_pred"}), on="listing_id")
)

summary = pd.DataFrame([
    {"model":"Random Forest", **metrics(all_pred["price"], all_pred["rf_pred"])},
    {"model":"Linear Regression", **metrics(all_pred["price"], all_pred["lr_pred"])},
    {"model":"Hybrid 50/50", **metrics(all_pred["price"], all_pred["hy_pred"])},
])

display(summary)
summary.to_csv(METRICS_DIR / "final_model_comparison.csv", index=False, encoding="utf-8-sig")

## Step 4 — Best Value per Metric

In [ ]:
rows = []
for m in ["MAE_THB","RMSE_THB","sMAPE_percent","MedianAE_THB"]:
    i = summary[m].idxmin()
    rows.append({"metric":m,"direction":"lower is better","model":summary.loc[i,"model"],"value":summary.loc[i,m]})

i = summary["R2"].idxmax()
rows.append({"metric":"R2","direction":"higher is better","model":summary.loc[i,"model"],"value":summary.loc[i,"R2"]})

leaders = pd.DataFrame(rows)
display(leaders)
leaders.to_csv(METRICS_DIR / "final_metric_leaders.csv", index=False, encoding="utf-8-sig")

## Step 5 — Error by Price Band

In [ ]:
bins = [-np.inf, 300_000, 600_000, 1_000_000, np.inf]
labels = ["<300k","300k-600k","600k-1m",">1m"]
all_pred["price_band"] = pd.cut(all_pred["price"], bins=bins, labels=labels)

rows = []
for band, g in all_pred.groupby("price_band", observed=True):
    for model_name, col in [
        ("Random Forest","rf_pred"),
        ("Linear Regression","lr_pred"),
        ("Hybrid 50/50","hy_pred"),
    ]:
        rows.append({"price_band":str(band),"model":model_name,"n":len(g),**metrics(g["price"],g[col])})

band_summary = pd.DataFrame(rows)
display(band_summary)
band_summary.to_csv(METRICS_DIR / "final_comparison_by_price_band.csv", index=False, encoding="utf-8-sig")

## Step 6 — Hybrid Coverage

In [ ]:
coverage = pd.DataFrame([{
    "test_rows": len(hy_df),
    "hybrid_used_rows": int(hy_df["hybrid_used"].astype(bool).sum()),
    "fallback_to_rf_rows": int((~hy_df["hybrid_used"].astype(bool)).sum()),
    "hybrid_coverage_percent": float(hy_df["hybrid_used"].astype(bool).mean()*100),
    "mean_comparable_count": float(hy_df["comparable_count"].mean()),
    "median_comparable_count": float(hy_df["comparable_count"].median()),
}])

display(coverage)
coverage.to_csv(METRICS_DIR / "final_hybrid_coverage.csv", index=False, encoding="utf-8-sig")

## Step 7 — Create Figures

In [ ]:
for metric, ylabel, filename in [
    ("MAE_THB","MAE (THB)","comparison_mae.png"),
    ("RMSE_THB","RMSE (THB)","comparison_rmse.png"),
    ("R2","R²","comparison_r2.png"),
    ("sMAPE_percent","sMAPE (%)","comparison_smape.png"),
]:
    plt.figure(figsize=(8,5))
    plt.bar(summary["model"], summary[metric])
    plt.ylabel(ylabel)
    plt.title(f"Model Comparison — {metric}")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=180)
    plt.show()

mae_band = band_summary.pivot(index="price_band", columns="model", values="MAE_THB").reindex(labels)
ax = mae_band.plot(kind="bar", figsize=(10,6))
ax.set_ylabel("MAE (THB)")
ax.set_xlabel("Actual price band")
ax.set_title("MAE by Price Band")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "comparison_mae_by_price_band.png", dpi=180)
plt.show()

## Step 8 — Save Unified Predictions

In [ ]:
final_predictions = (
    test_df[["listing_id","brand","model","sub_model","model_year","mileage","price"]]
    .merge(all_pred[["listing_id","rf_pred","lr_pred","hy_pred"]], on="listing_id", how="left")
)

for name, col in [("rf","rf_pred"),("lr","lr_pred"),("hybrid","hy_pred")]:
    final_predictions[f"{name}_abs_error"] = (final_predictions["price"] - final_predictions[col]).abs()

final_predictions.to_csv(
    PRED_DIR / "final_all_model_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

display(final_predictions.head())

## Step 9 — Final JSON Summary

In [ ]:
final_json = {
    "dataset":"One2Car Thailand",
    "target":"listing price",
    "currency":"THB",
    "test_rows":int(len(test_df)),
    "models":{
        row["model"]:{
            "MAE_THB":float(row["MAE_THB"]),
            "RMSE_THB":float(row["RMSE_THB"]),
            "R2":float(row["R2"]),
            "sMAPE_percent":float(row["sMAPE_percent"]),
            "MedianAE_THB":float(row["MedianAE_THB"]),
        }
        for _, row in summary.iterrows()
    },
    "hybrid_coverage_percent":float(coverage.loc[0,"hybrid_coverage_percent"]),
    "note":"Target is listing price, not confirmed transaction price."
}

with open(METRICS_DIR / "final_project_summary.json","w",encoding="utf-8") as f:
    json.dump(final_json,f,ensure_ascii=False,indent=2)

print(json.dumps(final_json,ensure_ascii=False,indent=2))